In [0]:
from typing import Any

In [0]:
import logging

from py4j.java_gateway import JavaObject  # type: ignore
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    coalesce,
    concat_ws,
    current_timestamp,
    from_utc_timestamp,
    last_day,
    lit,
    sha2,
    substring_index,
    when,
    regexp_extract,
    rank,
    desc,
    col,
    substring,
    length,
    concat,
    locate,
    upper,
    expr,
    regexp_replace,
    trim
)
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text('curated_zone', 'None', '01-Curated_zone')
dbutils.widgets.text('catalog_landed', 'None', '02-Catalog_landed')
dbutils.widgets.text('reference_zone', 'None', '03-Reference_zone')
dbutils.widgets.text('run_date', 'None', '04-Run Date')
dbutils.widgets.text('schema', 'None', '05-Schema')
dbutils.widgets.text('env_prefix', 'None', '06-Environment')
curated_zone = dbutils.widgets.get('curated_zone')
catalog_landed = dbutils.widgets.get('catalog_landed')
reference_zone = dbutils.widgets.get('reference_zone')
run_date = dbutils.widgets.get('run_date')
schema = dbutils.widgets.get('schema')
env_prefix = dbutils.widgets.get('env_prefix')

import sys, os
relative_path = '../../helpers/lng_lnd'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from landing_helpers import *
from landed_helpers import *
relative_path = '../../helpers/curated_helpers'
path = os.path.abspath(relative_path)
if path not in sys.path:
  sys.path.append(path)
from delta_table_creator import *
from extract import *
from transform import *
spark = SparkSession.builder.getOrCreate()


In [0]:
def load_curated_collection_process(
    spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix
) -> None:
    args = [curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix]
    if all(item is not None for item in args) :
        date_filter = f"businessdate = {run_date!r}"

        schema = (
            "CollectionId String, SourceSystem String, PortfolioKey int, PortfolioName String, IngestionTimestamp timestamp,"
            "BatchId String, CollectionKey String, CollectionSourceKey String, ExposureKey String, CepalKey String, Amount decimal(19,4), ReportingDate timestamp,"
            "RecordDate date, ActualDate date, ReverseCode String, Reverse String, CurrencyCode String, Currency String,"
            "LoanBankCode String, MatchedDate timestamp, Comments string, SplitFlag boolean, DiasNumber string, DiasChannel int,"
            "ExternalPaymentFlag boolean, ITReason string, FinalReason string, DoNotUpdateFlag boolean," 
            "PaymentFileChannel string, TanCode string, BranchCode string, CancellationDate timestamp, TaxNumber string, CDI string, BusinessDate date"
        )

        cluster_columns = ["BusinessDate"]

        curated_table_path = f"{curated_zone}.Collection"

        # ================== Create Delta Table ==================
        logging.info("Creating Delta Table...")
        if spark.catalog.tableExists(curated_table_path) is False:
            create_delta_table(
                spark,
                curated_table_path,
                cluster_columns,
                schema,  # type: ignore
            )
        else:
            logging.info("Table already exists.")

        # ================== Delete From Delta Table ==================
        if spark.catalog.tableExists(curated_table_path) is True:
            delete_from_curated_table(
                    spark,
                    curated_table_path,
                    run_date,  # type: ignore
                    "process",
                )        
        else:
            logging.info("Tables does not exist, no need to delete entries for business date!")

        # ================== portfolio ==================

        portfolio_extra_filters = None

        if portfolio_extra_filters is not None:
            portfolio_extra_filters.append(date_filter)
            portfolio_final_filters = " AND ".join(portfolio_extra_filters)
        else:
            portfolio_final_filters = date_filter  # possible need change date

        portfolio_selected_fields = [
            "PortfolioName",
            "Prefix",
            "unit_code_profits",
            "businessdate",
        ]

        portfolio = load_landed_portfolio_table(  # noqa: F841
            spark,
            curated_zone,  # type: ignore
            "portfolio",
            portfolio_final_filters,
            portfolio_selected_fields,
        )

        # ================== OpalDailyFilePaymentDetail ==================
        opap_daily_extra_filters = None

        if opap_daily_extra_filters is not None:
            opap_daily_extra_filters.append(date_filter)
            opap_daily_extra_filters = " AND ".join(opap_daily_extra_filters)
        else:
            opap_daily_final_filters = date_filter  # possible need change date

        opap_daily_selected_fields = [
            "DailPm_Key",
            "DailPm_OpapBranchCode",
            "DailPm_CancellationDateTime",
            "DailPm_TanNumber",
            "DailPm_TaxNumber",
            "DailPm_CDI",
            "businessdate",
        ]

        opap_daily_payments = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore #catalog_name instead
            "process_OpapDailyFilePaymentDetail",
            opap_daily_final_filters,
            opap_daily_selected_fields,
        )

        # ================== ProfitsCustomerRFCodes ==================
        profits_customer_extra_filters = None

        if profits_customer_extra_filters is not None:
            profits_customer_extra_filters.append(date_filter)
            profits_customer_extra_filters = " AND ".join(
                profits_customer_extra_filters
            )
        else:
            profits_customer_final_filters = date_filter  # possible need change date

        profits_customer_selected_fields = [
            "CUSTOMER_ID_NO",
            "CUST_RF_ID",
        ]

        profit_customer_rf = load_landed_delta_table(  # noqa: F841
            spark,
            catalog_landed,  # type: ignore #catalog_name instead!
            "profits_customer_rf_codes",
            profits_customer_final_filters,
            profits_customer_selected_fields,
        )

        # ================== tun_excluded ==================

        tun_excluded_extra_filters = None

        if tun_excluded_extra_filters is not None:
            tun_excluded_extra_filters.append(date_filter)
            tun_excluded_final_filters = " AND ".join(tun_excluded_extra_filters)
        else:
            tun_excluded_final_filters = date_filter

        tun_excluded_selected_fields = [
            "TUN",
        ]

        tun_excluded = load_landed_specificationparams_table( 
            spark,
            reference_zone,  # type: ignore
            "domus_excluded_tun_ask3423",
            tun_excluded_selected_fields,
        )


        # ================== collection_staging ==================

        collection_staging_extra_filters = None

        if collection_staging_extra_filters is not None:
            collection_staging_extra_filters.append(date_filter)
            collection_staging_final_filters = " AND ".join(
                collection_staging_extra_filters
            )
        else:
            collection_staging_final_filters = date_filter

        collection_staging_selected_fields = [
            "CepalID",
            "CepalIDUnmatched",
            "ID",
            "Loanbankcode",
            "Amount",
            "RecordDate",
            "ActualDate",
            "Reason",
            "ITReason",
            "Portfolio",
            "ReversedID",
            "ExternalPaymentFlag",
            "Comments",
            "CEPALPortfolioCode",
            "Currency",
            "RemittanceInfo",
            "CompanyID",
            "PaymentFileChannel",
            "IsModified",
            "BusinessDate"
        ]

        collection_staging = load_landed_delta_table(
            spark,
            catalog_landed,  # type: ignore
            "process_collection_stagingdata",
            collection_staging_final_filters,
            collection_staging_selected_fields,
        )
        
        # Replace non-breaking spaces and trim for loanbankcode
        collection_staging = collection_staging.withColumn(
            "Loanbankcode", trim(regexp_replace(col("Loanbankcode"), '[\r\n\u00A0]', '')))
        
        # ================== collection_previous ==================
        
        collection_previous_extra_filters = None

        collection_previous_selected_fields = [
            "PortfolioKey",
            "CollectionKey",
            "CollectionSourceKey",
            "CepalKey",
            "Amount",
            "RecordDate",
            "ActualDate",
            "Reverse",
            "Currency",
            "LoanBankCode",
            "MatchedDate",
            "Comments",
            "SplitFlag",
            "DiasNumber",
            "ExternalPaymentFlag",
            "ITReason",
            "FinalReason",
            "DoNotUpdateFlag",
            "PaymentFileChannel",
            "DiasChannel",
        ]

        collection_previous = load_previous_businessdate_table( 
            spark,
            curated_zone,  # type: ignore
            "collection",
            collection_previous_extra_filters,
            run_date,
            collection_previous_selected_fields
        )

        collection_previous = collection_previous.withColumn(
        "PaymentFileChannel",
        when(col("PaymentFileChannel") == "ΟΠΑΠ Πληρωμές", "OPAP")
        .when(col("PaymentFileChannel") == "Symbol Πληρωμές", "NBG")
        .when(col("PaymentFileChannel") == "AB Πληρωμές", "AB")
        .when(col("PaymentFileChannel") == "DIAS Πληρωμές", "DIAS")
        .otherwise(col("PaymentFileChannel"))
        )

        # ================== DiasDailyFilePaymentDetail ==================
        dias_daily_extra_filters = None

        if dias_daily_extra_filters is not None:
            dias_daily_extra_filters.append(date_filter)
            dias_daily_extra_filters = " AND ".join(
                dias_daily_extra_filters
            )
        else:
            dias_daily_final_filters = date_filter 

        dias_daily_selected_fields = [
            "DailPm_Tun",
            "DailPm_Channel",
        ]

        dias_daily = load_landed_delta_table(
            spark,
            catalog_landed, # type: ignore
            "cepalplatformportpayment_DiasDailyFilePaymentDetail",
            dias_daily_final_filters,
            dias_daily_selected_fields,
        )

        # ================== Start of Collections Preprocessing  ==================

        # ================== Collections Preprocessing Step 1: IsModified deduplication  ==================
        
        # For each Collection ID that has both IsModified=0 and IsModified=1 keep only the IsModified=1 rows
        ismodified_window = Window.partitionBy("ID").orderBy(
            desc(coalesce(col("IsModified"), lit(False)))
        )

        staging_deduped = (
            collection_staging
            .withColumn("rn", rank().over(ismodified_window))
            .where(col("rn") == 1)
            .drop("rn")
        )

        # ================== Collections Preprocessing Step 2: collection_staging enrichment  ==================
        
        staging_enriched = (
            staging_deduped.alias("s")
            .join(
                dias_daily.alias("d"), # Join with dias_daily
                col("s.ID") == col("d.DailPm_Tun"),
                "left"
            )
            .join(
                portfolio.alias("p"), # Join with portfolio p on CEPALPortfolioCode
                col("p.Prefix") == when(
                    length(col("s.CEPALPortfolioCode")) == 2,
                    concat(col("s.CEPALPortfolioCode"), lit("R"))
                ).otherwise(col("s.CEPALPortfolioCode")),
                "left"
            )
            .join(
                portfolio.alias("p2"), # Join with portfolio p2 with LoanBankCode prefix
                col("p2.Prefix") == substring(trim(col("s.LoanBankCode")), 1, 3),
                "left"
            )
        )

        # ================== Collections Preprocessing Step 3: Split matched and not matched  ==================

        matched_staging = staging_enriched.where(col("s.Reason") == "Matched")
        not_matched_staging = staging_enriched.where(
            coalesce(col("s.Reason"), lit("Not Matched")) == "Not Matched"
        )

        # ================== Collections Preprocessing Step 4: Join previous businessdate  ==================

        # Join on the natural key for matched
        matched_enriched = matched_staging.join(
            collection_previous.alias("prev"),
            on=[
                concat(col("s.CepalID").cast("string"), lit("_"), col("s.CepalIDUnmatched").cast("string")) == col("prev.CepalKey"),
                col("s.ExternalPaymentFlag") == col("prev.ExternalPaymentFlag"),
                col("s.PaymentFileChannel") == col("prev.PaymentFileChannel"),
                coalesce(col("p2.unit_code_profits"), col("p.unit_code_profits")) == col("prev.PortfolioKey")
            ],
            how="left"
        )

        # Join on the natural key for not matched
        not_matched_enriched = not_matched_staging.join(
            collection_previous.alias("prev"),
            on=[
                concat(col("s.CepalID").cast("string"), lit("_"), col("s.CepalIDUnmatched").cast("string")) == col("prev.CepalKey"),
                col("s.ExternalPaymentFlag") == col("prev.ExternalPaymentFlag"),
                col("s.PaymentFileChannel") == col("prev.PaymentFileChannel"),
                col("p.unit_code_profits") == col("prev.PortfolioKey")
            ],
            how="left"
        )

        # ================== Collections Preprocessing Step 5: Union Match and Unmatched  ==================
        
        all_enriched = matched_enriched.withColumn(
            "is_matched", lit(True)
        ).union(
            not_matched_enriched.withColumn(
                "is_matched", lit(False)
            )
        )

        # ================== Collections Preprocessing Step 6: Derive column values  ==================

        collection_payments = all_enriched.select(
        concat(col("s.CepalID").cast("string"), lit("_"), col("s.CepalIDUnmatched").cast("string")).alias("CepalKey"),
        col("s.ID").alias("CollectionSourceKey"),
        col("s.Amount"),
        col("s.RecordDate").alias("RecordDate"),
        col("s.ActualDate").alias("ActualDate"),
        col("s.ITReason").alias("ITReason"),
        col("s.ExternalPaymentFlag"),
        col("s.Comments"),
        col("s.ReversedID").alias("Reverse"),
        col("s.Currency"),
        col("s.RemittanceInfo"),
        col("s.CompanyID"),
        col("s.PaymentFileChannel"),
        col("s.IsModified"),
        col("s.LoanBankCode").alias("LoanBankCode"),
        col("d.DailPm_Channel").alias("DiasChannel"),
        when(col("s.PaymentFileChannel") == "DIAS", col("s.RemittanceInfo"))
        .when(
            col("s.RemittanceInfo").like("DIASDEBIT%"),
                trim(regexp_extract(col("s.RemittanceInfo"), "(RF.*)", 1))
        )
        .otherwise(lit(None))
        .alias("DiasNumber"),

        when(col("is_matched"), lit("Matched"))
        .otherwise(lit("Not Matched"))
        .alias("FinalReason"), #different for matched and unmatched

        when(
            col("is_matched"),
            coalesce(col("prev.PortfolioKey"), col("p2.unit_code_profits"), col("p.unit_code_profits"))
        ).otherwise(
            coalesce(col("prev.PortfolioKey"), col("p.unit_code_profits"))
        ).alias("PortfolioKey"), #different for matched and unmatched

        when(
            ~col("is_matched"),
            lit(None).cast("date")
        ).when(
            coalesce(col("prev.FinalReason"), lit("Not Matched")) == "Not Matched",
            lit(run_date)
        ).when(
            col("prev.MatchedDate") < col("s.ActualDate"),
            date_add(col("s.ActualDate"), 1)
        ).otherwise(
            coalesce(col("prev.MatchedDate"), lit(run_date))
        ).alias("MatchedDate"), #different for matched and unmatched

        coalesce(
            col("prev.CollectionKey"),
            upper(substring(
                sha2(
                    concat_ws(";",
                        col("s.CepalID").cast("string"),
                        col("s.CepalIDUnmatched").cast("string"),
                        col("s.ExternalPaymentFlag").cast("string"),
                        col("s.PaymentFileChannel"),
                        coalesce(col("p2.unit_code_profits"), col("p.unit_code_profits")).cast("string")
                    ),
                    256
                ),
                1, 32
            ))
        ).alias("CollectionKey"),
        lit(False).alias("SplitFlag"),
        lit(None).alias("DoNotUpdateFlag"),
        col("s.BusinessDate")
        )

        # ================== Collections Preprocessing Step 7: Preserve split payments collections ==================

        #Add the current businessdate
        previous_splits = (
        collection_previous
        .where(col("SplitFlag") == True)
        .withColumn("BusinessDate", lit(run_date))
        )
        
        # Find CollectionSourceKey and PortfolioKey combinations that have split collections
        split_ids = (
            previous_splits
            .select("CollectionSourceKey", "PortfolioKey")
            .distinct()
        )

        # Find the CollectionKeys of unsplit original collections
        originals_to_remove = (
            collection_payments.alias("t")
            .join(
                split_ids.alias("sp"),
                (col("t.CollectionSourceKey") == col("sp.CollectionSourceKey")) & (col("t.PortfolioKey") == col("sp.PortfolioKey")),
                "inner"
            )
            .where(coalesce(col("t.SplitFlag"), lit(False)) == False)
            .select(col("t.CollectionKey"))
            .distinct()
        )

        # Remove unsplit original collections
        collection_payments = collection_payments.join(
            originals_to_remove,
            on="CollectionKey",
            how="left_anti"
        )

        #Add the splits back
        collection_payments = collection_payments.unionByName(
            previous_splits,
            allowMissingColumns=True
        )        

        # ================== End of Collections Preprocessing ==================

        # HashKeys
        collection_payments_primary_key_fields = [
            collection_payments.CollectionKey,
            lit("process"),
        ]

        ##exclude all the TUNs that are in the tun_excluded table
        collection_payments = collection_payments.join(
            tun_excluded,
            collection_payments.CollectionSourceKey == tun_excluded.TUN,
            how='left_anti'
        )

        collection_payments = (
            collection_payments.withColumn(
                "CollectionId",
                sha2(concat_ws(";", *collection_payments_primary_key_fields), 256),
            )
            .withColumn(
                "ReportingDate",
                when(
                    collection_payments.MatchedDate.isNull(),
                    collection_payments.ActualDate,
                )
                .when(
                    last_day(collection_payments.ActualDate)
                    != last_day(collection_payments.MatchedDate),
                    collection_payments.MatchedDate,
                )
                .otherwise(collection_payments.ActualDate),
            )
            .withColumn(
                "PaymentFileChannel",
                when(collection_payments.PaymentFileChannel == "OPAP", "ΟΠΑΠ Πληρωμές")
                .when(
                    collection_payments.PaymentFileChannel == "NBG", "Symbol Πληρωμές"
                )
                .when(collection_payments.PaymentFileChannel == "AB", "AB Πληρωμές")
                .when(collection_payments.PaymentFileChannel == "DIAS", "DIAS Πληρωμές")
                .otherwise(collection_payments.PaymentFileChannel),
            )
        )

        collection_payments_portfolio_join_condition = (
            collection_payments["PortfolioKey"] == portfolio["unit_code_profits"]
        )

        # Join condition for collection_payments and opapdaily
        collection_payments_opapdaily_join_condition = (
            substring_index(collection_payments["CepalKey"], "_", 1)
            == opap_daily_payments["DailPm_Key"]
        ) & (collection_payments["PaymentFileChannel"] == "ΟΠΑΠ Πληρωμές")

        # Join condition for collection_payments and profit_customer_rf
        profit_customer_rf_join_condition = (
            collection_payments["DiasNumber"] == profit_customer_rf["CUST_RF_ID"]
        )

        target_table_profits = (
            collection_payments.join(
                portfolio,
                collection_payments_portfolio_join_condition,
                "left",
            )
            .join(
                opap_daily_payments,
                collection_payments_opapdaily_join_condition,
                "left",
            )
            .join(
                profit_customer_rf,
                profit_customer_rf_join_condition,
                "left",
            )
        )

        # ================== Selection of Final Columns ==================
        result_df = target_table_profits.select(
            collection_payments.CollectionId.cast("string").alias("CollectionId"),
            lit("process").cast("string").alias("SourceSystem"),
            collection_payments.PortfolioKey.cast("int").alias("PortfolioKey"),
            portfolio.PortfolioName.cast("string").alias("PortfolioName"),
            lit(from_utc_timestamp(current_timestamp(), "Europe/Athens")).cast("timestamp").alias(
                "IngestionTimestamp"
            ),
            lit(None).cast("string").alias("BatchId"),
            collection_payments.CollectionKey.cast("string").alias("CollectionKey"),
            collection_payments.CollectionSourceKey.cast("string").alias("CollectionSourceKey"),
            collection_payments.LoanBankCode.cast("string").alias("ExposureKey"),
            collection_payments.CepalKey.cast("string").alias("CepalKey"),
            collection_payments.Amount.cast("decimal(19,4)").alias("Amount"),
            collection_payments.ReportingDate.cast("timestamp").alias("ReportingDate"),
            collection_payments.RecordDate.cast("date").alias("RecordDate"),
            collection_payments.ActualDate.cast("date").alias("ActualDate"),
            collection_payments.Reverse.cast("string").alias("ReverseCode"),
            collection_payments.Reverse.cast("string").alias("Reverse"),
            collection_payments.Currency.cast("string").alias("CurrencyCode"),
            collection_payments.Currency.cast("string").alias("Currency"),
            collection_payments.LoanBankCode.cast("string").alias("LoanBankCode"),
            collection_payments.MatchedDate.cast("timestamp").alias("MatchedDate"),
            collection_payments.Comments.cast("string").alias("Comments"),
            collection_payments.SplitFlag.cast("boolean").alias("SplitFlag"),
            collection_payments.DiasNumber.cast("string").alias("DiasNumber"),
            collection_payments.DiasChannel.cast("int").alias("DiasChannel"),
            collection_payments.ExternalPaymentFlag.cast("boolean").alias("ExternalPaymentFlag"),
            collection_payments.ITReason.cast("string").alias("ITReason"),
            collection_payments.FinalReason.cast("string").alias("FinalReason"),
            collection_payments.DoNotUpdateFlag.cast("boolean").alias("DoNotUpdateFlag"),
            collection_payments.PaymentFileChannel.cast("string").alias("PaymentFileChannel"),
            opap_daily_payments.DailPm_TanNumber.cast("string").alias("TanCode"),
            opap_daily_payments.DailPm_OpapBranchCode.cast("string").alias("BranchCode"),
            opap_daily_payments.DailPm_CancellationDateTime.cast("timestamp").alias("CancellationDate"),
            opap_daily_payments.DailPm_TaxNumber.cast("string").alias("TaxNumber"),
            coalesce(
                opap_daily_payments.DailPm_CDI, profit_customer_rf.CUSTOMER_ID_NO
            ).cast("string").alias("CDI"),
            collection_payments.BusinessDate.cast("date").alias("BusinessDate"),
        )

        # ================== Append ==================

        result_df.write.format("delta").clusterBy("BusinessDate").mode("append").saveAsTable(
            curated_table_path
        )
    else:
        null_args = [
            arg_name
            for arg_name, arg in args
            if arg
            in [
                curated_zone,
                catalog_landed,
                reference_zone,
                run_date,
                schema,
                env_prefix,
            ]
            and arg is None
        ]

        for arg_name in null_args:
            logging.error(f"No {arg_name} specified!")

    logging.info("We did it!")


In [0]:
load_curated_collection_process(spark, curated_zone, catalog_landed, reference_zone, run_date, schema, env_prefix)